In [2]:
import os
import numpy as np
from datetime import datetime as dt
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from sb3_contrib import TRPO
from rlquantopt.rl_envs.zc_qpee import ZCQPEE
from rlquantopt.rl_envs.zcqubits import setup_ZCQubits4MKrauss_params
from tqdm.notebook import tqdm

In [15]:
# best agent trained with static model params
model_zip = '/home/leander/code/rlquantopt/rlquantopt/rl_agents/ZCQPEE_pl-1000_T-50ns_delta_mode-TRPO/05-12-24_201634/rl_model_12566528_steps.zip'
# best agent trained with +-0.1% model param drift
# model_zip = '/home/leander/code/rlquantopt/rlquantopt/rl_agents/ZCQPEEWRD_pl-1000_T-50ns_delta_mode-TRPO/25-03-25_115755/rl_model_19382272_steps.zip'
save_dir = '.'
print(f"Saving results to: {os.path.abspath(save_dir)}")

model_dir = os.path.dirname(model_zip)

n_eps = 1

algo_str = 'TRPO'
algo = TRPO

model_name = os.path.splitext(os.path.basename(model_zip))[0]
model = algo.load(model_zip)
model

Saving results to: /home/leander/code/rlquantopt/rlquantopt/rl_analysis/generalisation


/home/leander/miniconda3/envs/qvaqt/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [21]:
from itertools import product

In [ ]:
model.policy.

In [3]:
n_params = 0
for p in model.policy.parameters():
    p_ = np.prod(p.size())
    print(p)
    print(p.size())
    # print(p_)
    n_params += p_
print(f"total nb. params = {n_params}")
    # print(p)

NameError: name 'model' is not defined

In [18]:
# save_dir = "sweep-fine_20250317_030211"
save_dir = None
# save_dir = "sweep-fine_20250401_004354"
if save_dir is None:
    save_dir = f"sweep-fine_{dt.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(save_dir, exist_ok=True)
if os.path.exists(save_dir):
    print(f"Experiment dir: {save_dir}")
else:
    raise FileNotFoundError

Experiment dir: sweep-fine_20250402_103407


In [21]:
_save_path = lambda x: os.path.join(save_dir, f'{x if x < max_it - 1 else "final"}')

def quick_plot(arrs, save_fn):
    best_reward_global = arrs[0].T
    best_concur_global = arrs[1].T
    best_unitar_global = arrs[2].T
    # best_indices = arrs[3].T
    
    fig, axs = plt.subplots(3, figsize=(8,15))
    x_ticks = np.linspace(0, len(ω_1_list)-1, 6).astype(int)#[0]- ω_1, ω_1_list[-1]- ω_1, 6)
    y_ticks = np.linspace(0, len(ω_2_list)-1, 6).astype(int)#[0]- ω_2, ω_2_list[-1]- ω_2, 6)
    # print(x_ticks)
    xtick_labels = [f'{(ω_1_list[i] - ω_1)*1000.:.2f}' for i in x_ticks]
    ytick_labels = [f'{(ω_2_list[i] - ω_2)*1000.:.2f}' for i in y_ticks]
    
    ax = axs[0]
    pos = ax.get_position()  # Get the current position
    ax.set_position([pos.x0, pos.y0, pos.width * 10., pos.height * 1.2])  # Expand width & height
    
    for ax, title, data, log_norm, a_min in zip(axs, 
                              ('Rewards', 'Concurrence Error', 'Unitarity Error'),#, 'Best Time'),
                              (best_reward_global, np.subtract(1, best_concur_global), np.subtract(1, best_unitar_global)),#, (best_indices+1)*0.05),
                              (False, True, True, False), (0, 1e-6, 1e-4, 0)):
        ax.axhline(np.searchsorted(ω_2_list, 0) , color='k', ls='dashed')
        ax.axvline(np.searchsorted(ω_1_list, 0) , color='k', ls='dashed')
        if log_norm:
            norm = LogNorm(vmin=a_min, vmax=1)
        else:
            norm = Normalize(vmin=a_min, vmax=4)
    
        data = np.clip(data, a_min=a_min, a_max=np.inf)
    
        ax.set_title(title)
        ax.set_aspect('auto')
        # pos = ax.get_position()  # Get the current position
        # ax.set_position([pos.x0, pos.y0, pos.width * 10, pos.height * 1.2])  # Expand width & height
    
        heatmap = ax.imshow(data, cmap='magma', norm=norm, aspect='auto', origin='lower')
        cbar = fig.colorbar(heatmap, ax=ax)
        # cbar.set_clim((0, 4))
    
        ax.set_xticks(x_ticks)
        ax.set_yticks(y_ticks)
        ax.set_xticklabels(xtick_labels)
        ax.set_yticklabels(ytick_labels)
        ax.set_xlabel(r'$\Delta ω_1$ [MHz]')
        ax.set_ylabel(r'$\Delta ω_2$ [MHz]')
    
    fig.tight_layout()
    fig.savefig(save_fn, dpi=500)
    plt.close(fig)
    print(f'FIGURE SAVED TO: {save_fn}')


In [16]:
model_params = setup_ZCQubits4MKrauss_params()
ω_1, ω_2 = model_params['omega_s']

max_deviation_percent_1 = 1
max_deviation_percent_2 = 1
n_intervals_1 = 100
n_intervals_2 = 100
ω_1_list = ω_1 * (1 + np.linspace(-max_deviation_percent_1/100., max_deviation_percent_1/100., n_intervals_1 + 1))
ω_2_list = ω_2 * (1 + np.linspace(-max_deviation_percent_2/100., max_deviation_percent_2/100., n_intervals_2 + 1))

print(f"Original ω_1={ω_1}\tSweep: {len(ω_1_list)} parameters")
print(f"Original ω_2={ω_2}\tSweep: {len(ω_2_list)} parameters")
print(f"Nb. searches = {len(ω_1_list) * len(ω_2_list)}")

Original ω_1=5.0311	Sweep: 101 parameters
Original ω_2=5.8899	Sweep: 101 parameters
Nb. searches = 10201


In [17]:
start_i1 = 100
max_it = np.inf
# start_i1 = 400

if start_i1 is not None:
    load_fn = _save_path(start_i1) + '.npy'
    best_global = np.load(load_fn)
    print(f"Loaded results shape={best_global.shape} from: {load_fn}")

Loaded results shape=(4, 101, 101) from: sweep-fine_20250401_004354/100.npy


In [9]:
import json

In [27]:
with open('sweep_max-dev-1.0_n-intervals-100.json', 'r') as f:
    best_global_dict = json.load(f)
best_global = []
for v in best_global_dict.values():
    best_global.append(np.transpose(v))
best_global = np.array(best_global)
best_global.shape

(3, 101, 101)

In [28]:
quick_plot(best_global, _save_path(start_i1)+'.pdf')

FIGURE SAVED TO: sweep-fine_20250402_103407/100.pdf
